In [2]:
# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

# Pick 9 patients with different complexity scores

In [3]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [4]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


# Load `chunks_df`

In [6]:
# look at all oncology chunks in chunks_df:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_ids = selected_patient_ids

conn = sqlite3.connect(db_path)

# Multiple patient_ids: build an IN (...) placeholder list.
if not patient_ids:
    chunks_df = pd.DataFrame()  # avoid invalid SQL: IN ()
else:
    placeholders = ",".join(["?"] * len(patient_ids))
    query = f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    """
    chunks_df = pd.read_sql_query(query, conn, params=patient_ids)

conn.close()

display(chunks_df.head())
print("Number of all chunks:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
1,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,cd9bf67f542dee2c5c6eb4e889086745d239ff7b,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
2,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,c6f4af530450ec4d38f7f478693b4d62c2b3467c,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
3,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,9d5bdbca525ce28a75b1ad7deff73853cc1b20eb,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
4,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,250fb82c086a015ec8fe85d5feeb46806e632e86,0,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of all chunks: 14390


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
21,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,34e083fbe248b34ab7ff75e989fc91da40a6b511,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
931,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,6c44d78bfe33c70133759592f49f88f9cd26734c,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
932,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,4f181909c2fad14ebbdc20f28f1af14d5dfb89f9,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
933,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,b02dbcb96bafeb63dc73b3c40a9f184cb0e781bf,1,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of oncology chunks: 890


# A high complexity patient

In [7]:
patient_id = "f203e11d-5573-1624-69b8-af8436987b3e" #high complexity patient

overview_question = "Provide a brief overview of this patient's medical background and current status."
conditions_question = "What are this patient's main diagnosed conditions and their status?"
medications_question = "What medications is this patient currently or recently taking?"
oncology_question = "Summarize this patient's oncology history."

In [8]:
import rag_service as rag

In [11]:
# helper function to print the answer and judge it

import json


def print_answer_and_judge(
    label: str,
    question: str,
    output: dict,
    search_type: str = "hybrid",
    judge_model: str = "gpt-5.4-mini",
) -> dict:
    print(f"\n=== {label} answer for {output['patient_name']} ===")
    print(output["answer"])

    context = (
        output.get("context")
        or output.get("retrieved_context")
        or output.get("retrieved_text")
    )

    if not context:
        print("\n--- Judge evaluation unavailable ---")
        print(
            "No retrieved context was found in the rag_new() output. "
            f"Available keys: {list(output.keys())}"
        )
        return output

    judge_out = rag.evaluate_relevance(
        question=question,
        answer=output["answer"],
        context=context,
        search_type=search_type,
        model=judge_model,
    )

    output["judge"] = judge_out

    print("\n--- Judge evaluation ---")
    # print(json.dumps(judge_out["evaluation"], indent=2))
    print(json.dumps(judge_out, indent=2))

    print("\n--- Judge usage and cost ---")
    print(json.dumps(judge_out["token_stats"], indent=2))
    print(json.dumps(judge_out["cost"], indent=2))

    return output

In [12]:
# Patient overview
overview_out = rag.rag_new(
    query=overview_question,
    patient_id=patient_id,
    question_type="patient_overview",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

print(
    f"=== Patient overview answer for {overview_out['patient_name']} "
    f"(DOB {overview_out['patient_dob']}, "
    f"Age: {overview_out['patient_age_years']} years, "
    f"Gender: {overview_out['patient_gender']}) ==="
)

overview_out = print_answer_and_judge(
    label="Patient overview",
    question=overview_question,
    output=overview_out,
)


# Conditions
conditions_out = rag.rag_new(
    query=conditions_question,
    patient_id=patient_id,
    question_type="conditions",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

conditions_out = print_answer_and_judge(
    label="Conditions",
    question=conditions_question,
    output=conditions_out,
)


# Medications
medications_out = rag.rag_new(
    query=medications_question,
    patient_id=patient_id,
    question_type="medications",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

medications_out = print_answer_and_judge(
    label="Medications",
    question=medications_question,
    output=medications_out,
)


# Oncology timeline
oncology_out = rag.rag_new(
    query=oncology_question,
    patient_id=patient_id,
    question_type="oncology_timeline",
    num_results=5,
    model="gpt-5.4-mini",
    search_type="hybrid",
)

oncology_out = print_answer_and_judge(
    label="Oncology timeline",
    question=oncology_question,
    output=oncology_out,
)

=== Patient overview answer for Shawana711 Lakin515 (DOB 1965-01-05, Age: 61 years, Gender: female) ===

=== Patient overview answer for Shawana711 Lakin515 ===
1. **Summary:**
- The documented clinical conditions include malignant neoplasm of breast, acute myeloid leukemia (resolved), hypoxemia, acute pulmonary embolism (resolved), pneumonia (resolved), viral sinusitis (resolved), sepsis caused by virus (resolved), and concussion with loss of consciousness (resolved).
- The most recent clinically important active condition documented is hypoxemia, and other major recent conditions are resolved.
- Oncology timeline shows breast cancer diagnosed in 2015 as stage 2/2A, HER2-negative, T2N0M0, with repeated completed radiotherapy course events and follow-up observations noting the patient's condition improved through 2019-03-22.

2. **Active Conditions:**
- **Malignant neoplasm of breast (disorder)** — active; date: 2015-05-07
- **Hypoxemia (disorder)** — active; date: 2020-05-04
- **Acute

In [ ]:
# # Patient overview
# overview_out = rag.rag_new(
#     query=overview_question,
#     patient_id=patient_id,
#     question_type="patient_overview",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(
#     f"=== Patient overview answer for {overview_out['patient_name']} "
#     f"(DOB {overview_out['patient_dob']}, Age: {overview_out['patient_age_years']} years, "
#     f"Gender: {overview_out['patient_gender']}) ==="
# )
# print(overview_out["answer"])

# # Conditions
# conditions_out = rag.rag_new(
#     query=conditions_question,
#     patient_id=patient_id,
#     question_type="conditions",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Conditions answer for patient {conditions_out['patient_name']} ===")
# print(conditions_out["answer"])

# # Medications
# medications_out = rag.rag_new(
#     query=medications_question,
#     patient_id=patient_id,
#     question_type="medications",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Medications answer for patient {medications_out['patient_name']} ===")
# print(medications_out["answer"])

# # Oncology timeline (if configured)
# oncology_out = rag.rag_new(
#     query=oncology_question,
#     patient_id=patient_id,
#     question_type="oncology_timeline",
#     num_results=5,
#     model="gpt-5.4-mini",
#     search_type="hybrid",
# )

# print(f"\n=== Oncology timeline answer for patient {oncology_out['patient_name']} ===")
# print(oncology_out["answer"])